In [ ]:
# RGTransformer 验证与分析
# 说明：用于模型验证、预测可视化、区域分析
# 训练请使用: python src/RGT.py

%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('X:/Workspace/3D-Ocean')

# 从配置文件导入（轻量级，不导入 PyTorch）
from src.trainer.config import (
    area, resolution, width, height,
    dataset_params, model_params
)
from src.trainer.base import BasePrediction
from src.models.SST.RGTransformer import RGTransformer
from src.dataset.OISST import OISSTMonthlyDataset
from src.config.params import PROJECT_PATH

print("✅ 库导入完成")
print(f"项目根目录: {PROJECT_PATH}")


In [ ]:
# %% 配置
# ⚠️ 只需修改 RUN_ID（训练时生成的时间戳）
# 其他配置自动从 src/trainer/config.py 导入

RUN_ID = "2026-01-02-20-16"  # 👈 修改为你的 run_id

print("=" * 70)
print("📋 配置（从 trainer/config.py 导入）")
print("=" * 70)
print(f"Run ID: {RUN_ID}")
print(f"区域: {area.title}")
print(f"分辨率: {resolution}°")
print(f"空间尺寸: {width} x {height}")
print("=" * 70)

In [ ]:
# %% 1. 加载模型并预测
# 从 checkpoint 加载模型，进行预测和可视化

print("=" * 70)
print("📊 加载模型")
print("=" * 70)

# 创建预测器
predictor = BasePrediction(
    run_id=RUN_ID,  # 优先从本地 out/checkpoints/{run_id}/ 加载
    area=area,
    model_class=RGTransformer,
    dataset_class=OISSTMonthlyDataset,
    dataset_params=dataset_params,
    model_params=model_params,
)

# 预测并可视化
TEST_OFFSET = 520  # 测试时间点

result = predictor.predict(offset=TEST_OFFSET, plot=True)
input_data, output_data, pred_output, rmse, r2, ssta = result

print(f"\n📈 预测结果:")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²: {r2:.4f}")
print("=" * 70)

In [ ]:
# %% 2. 区域性分析
# 对关键海洋区域进行针对性分析

from src.analysis.regional import run_regional_analysis

print("=" * 70)
print("🌊 区域性分析")
print("=" * 70)

# 运行区域分析
# detail_regions 可选值：
#   'nino34'（厄尔尼诺监测区）, 'nino3'（东太平洋暖池）, 
#   'warm_pool'（赤道太平洋暖池）, 'gulf_stream'（墨西哥湾暖流）,
#   'kuroshio'（黑潮）, 'acc'（南大洋西风漂流）, 
#   'north_indian'（北印度洋）, 'north_atlantic_subpolar'（北大西洋副极地）

analyzer, stats, summary_df = run_regional_analysis(
    predictor=predictor,
    area=area,
    resolution=resolution,
    test_offset=TEST_OFFSET,
    save_dir='out/sst/regional',
    detail_regions=['nino34', 'gulf_stream', 'kuroshio', 'acc', 'warm_pool'],
    show_plots=True
)

print("\n✅ 区域分析完成！")
